# Actividad5_Visualizacion_Resultados

**Actividad 5, Semana 9:** Técnicas de visualización de resultados en Big Data  
**Autor:** Jonathan Javier Monsalve Giraldo  
**Matrícula:** A01840272  
**Proyecto base:** NYC TLC Yellow Taxi, Proyecto Etapa 3, Equipo 12  
**Preguntas de aprendizaje:** predicción supervisada de `fare_amount` y segmentación no supervisada de arquetipos operativos  
**Modelos base:** `RandomForestRegressor` ganador supervisado de Etapa 3 (`numTrees=50`, `maxDepth=10`, `subsamplingRate=1.0`) y KMeans ganador no supervisado de Etapa 3 (`KMEANS_CLUSTERS = 5`)

El objetivo de este notebook es construir una validación cruzada k-fold sobre la muestra M definida en el proyecto y visualizar la variabilidad de los resultados entre pliegues para las dos preguntas de aprendizaje heredadas del proyecto. La primera pregunta estima `fare_amount` con un modelo supervisado de regresión; la segunda identifica arquetipos operativos de viaje con un modelo no supervisado de clustering. Los modelos no compiten entre sí porque responden preguntas distintas y se evalúan con métricas diferentes.

## Secciones de la rúbrica

La sección 0 reconstruye M para que el notebook sea autocontenido. Las cinco secciones evaluables de la rúbrica son las siguientes y aparecen con estos títulos exactos más abajo:

1. [Cálculo del valor K para validación cruzada](#1-cálculo-del-valor-k-para-validación-cruzada)
2. [Construcción de los k-folds](#2-construcción-de-los-k-folds)
3. [Experimentación con Validación Cruzada](#3-experimentación-con-validación-cruzada)
4. [Resultados de la Validación Cruzada](#4-resultados-de-la-validación-cruzada)
5. [Discusión y conclusiones](#5-discusión-y-conclusiones)

En Etapa 3 se reportaron dos modelos ganadores para dos preguntas distintas: `RandomForestRegressor` para predecir `fare_amount` y KMeans con `KMEANS_CLUSTERS = 5` para descubrir arquetipos operativos. La validación cruzada de esta actividad reutiliza la misma muestra M para evaluar estabilidad, variabilidad e interpretabilidad en ambos frentes.


# 0. Configuración y reconstrucción de la muestra M

Esta sección reconstruye la muestra M con el mismo procedimiento usado en Etapa 3: descarga y carga de datos, limpieza, imputaciones, variables de caracterización, cálculo de fracciones por estrato, extracción de M y validación de representatividad marginal. No se reproduce el split fijo 80/20 de Etapa 3 porque esta actividad usa validación cruzada: M se divide en k folds, y en cada iteración un fold funciona como prueba mientras los k-1 folds restantes funcionan como entrenamiento.


Esta celda instala o verifica dependencias, importa las bibliotecas necesarias e inicializa Spark con la configuración de memoria usada en Etapa 3. Se conserva `WARN` global y se filtra solo el aviso repetitivo `Broadcasting large task binary`, que no afecta la interpretación de los resultados.


In [ ]:
# Librerías estándar
import importlib.util
import subprocess
import sys
import time
import urllib.request
from functools import reduce
from pathlib import Path

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # En Colab, Java suele estar disponible. Esta verificación deja explícito el requisito.
    java_check = subprocess.run(["bash", "-lc", "java -version"], capture_output=True, text=True)
    if java_check.returncode != 0:
        subprocess.check_call(["apt-get", "update"])
        subprocess.check_call(["apt-get", "install", "-y", "openjdk-11-jdk-headless"])

ensure_package("findspark")
ensure_package("pyspark")
ensure_package("pandas")
ensure_package("numpy")
ensure_package("matplotlib")
ensure_package("seaborn")

# Librerías externas
import findspark
findspark.init()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator, RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

DATA_DIR = Path("data") / "raw"

spark = (
    SparkSession.builder
    .appName("Actividad5_Visualizacion_Resultados")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.debug.maxToStringFields", "200")
    .getOrCreate()
)


spark.sparkContext.setLogLevel("WARN")
log4j = spark.sparkContext._jvm.org.apache.logging.log4j
log4j.LogManager.getLogger("org.apache.spark.scheduler.DAGScheduler").addFilter(spark.sparkContext._jvm.org.apache.logging.log4j.core.filter.RegexFilter.createFilter(".*Broadcasting large task binary.*", None, True, log4j.core.Filter.Result.DENY, log4j.core.Filter.Result.NEUTRAL))
sns.set_theme(style="whitegrid")

print(f"Entorno Colab: {IN_COLAB}")
print(f"Ruta de datos: {DATA_DIR}")
print(f"Versión de Spark: {spark.version}")


## 0.1 Descarga reproducible de datos

La celda siguiente descarga los 24 archivos Parquet de 2024 y 2025, además del catálogo de zonas. La descarga es idempotente: si un archivo ya existe en `data/raw`, no se vuelve a descargar.


Esta celda descarga los archivos de NYC TLC y el catálogo de zonas solo si no existen localmente. Si ya están en `data/raw`, la descarga se omite.


In [ ]:
# Ejecutar esta celda manualmente en Jupyter/Colab para descargar los datos.
CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)


def fetch(url, target):
    target = Path(target)

    if target.exists():
        return "skip"

    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)
    return "downloaded"


download_log = []
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        status = fetch(f"{CDN_BASE}/trip-data/{filename}", DATA_DIR / filename)
        download_log.append((filename, status))

zones_status = fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")

print("Archivos Parquet revisados:", len(download_log))
print("Archivos descargados:", sum(1 for _, status in download_log if status == "downloaded"))
print("Catálogo de zonas:", zones_status)


## 0.2 Carga, downcast y limpieza cerrada

Se replica la limpieza cerrada de Etapa 3. La población analítica conserva los viajes de 2024 y 2025 que cumplen los filtros de calidad previamente definidos por el equipo.


Esta celda carga los Parquet, aplica downcast de tipos y reproduce los filtros cerrados de calidad definidos en el proyecto.


In [ ]:
parquet_paths = sorted(str(path) for path in DATA_DIR.glob("yellow_tripdata_*.parquet"))
assert len(parquet_paths) == 24, f"Se esperaban 24 archivos Parquet, se encontraron {len(parquet_paths)}"

df_native = spark.read.option("mergeSchema", "true").parquet(*parquet_paths)
zones = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv"))
)

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (
    df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0)))
)

n_raw = df_raw.count()
n_filtered = df_filtered.count()
pct_removed = (n_raw - n_filtered) / n_raw * 100

print(f"Filas crudas: {n_raw:,}")
print(f"Filas tras limpieza: {n_filtered:,}")
print(f"Pérdida por filtros cerrados de Etapa 2: {pct_removed:.2f}%")

assert n_raw == 89_892_322, "El conteo crudo no coincide con la referencia de Etapa 1"
assert abs(pct_removed - 6.07) < 0.20, "La pérdida por limpieza difiere de la referencia de Etapa 2"


## 0.3 Imputaciones operativas

Las imputaciones replican Etapa 3: se corrigen nulos estructurales y valores fuera de dominio sin descartar filas adicionales. La bandera `is_flex_fare` se conserva para distinguir el régimen Flex.


Esta celda aplica las imputaciones operativas heredadas de Etapa 3 y verifica que no queden nulos en las columnas corregidas.


In [ ]:
df_clean = (
    df_filtered
    .withColumn(
        "passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
        .otherwise(F.lit(1).cast("byte")),
    )
    .withColumn(
        "cbd_congestion_fee",
        F.when(
            F.col("cbd_congestion_fee").isNull()
            | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
            F.lit(0.0).cast("float"),
        ).otherwise(F.col("cbd_congestion_fee")),
    )
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F")))
)

imputed_cols = [
    "passenger_count",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "Airport_fee",
    "RatecodeID",
    "store_and_fwd_flag",
]

null_row = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
null_summary = [(c, int(null_row[c] or 0)) for c in imputed_cols]

print("Nulos remanentes tras imputación:")
for col_name, n_nulls in null_summary:
    print(f"{col_name}: {n_nulls:,}")

assert all(n_nulls == 0 for _, n_nulls in null_summary), "Quedan nulos en columnas imputadas"


## 0.4 Variables de caracterización y extracción de M

La muestra M se estratifica con cuatro variables: macrozona de origen, grupo de pago, bloque horario y rango de distancia. Su concatenación forma `stratum_id`, la unidad de muestreo y partición.


Esta celda construye las variables de caracterización que definen los estratos de M: macrozona, grupo de pago, bloque horario y rango de distancia.


In [ ]:
airport_ids = {row.LocationID for row in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {row.LocationID for row in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {row.LocationID for row in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

assert not (airport_ids & manhattan_ids)
assert not (airport_ids & outer_ids)
assert not (unknown_ids & manhattan_ids)
assert not (unknown_ids & outer_ids)

df_feat = (
    df_clean
    .withColumn(
        "pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
        .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
        .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
        .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
        .otherwise("unknown"),
    )
    .withColumn(
        "payment_group",
        F.when(F.col("payment_type") == 0, "flex")
        .when(F.col("payment_type") == 1, "credit")
        .when(F.col("payment_type") == 2, "cash")
        .otherwise("other"),
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn(
        "day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
        .when(F.col("dow").isin(1, 7), "weekend")
        .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
        .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
        .otherwise("other"),
    )
    .withColumn(
        "trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
        .when(F.col("trip_distance") < 12.43, "medium")
        .otherwise("long"),
    )
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn(
        "cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd")
        .otherwise("post_cbd"),
    )
    .withColumn(
        "stratum_id",
        F.concat_ws(
            "|",
            F.col("pu_macro_zone"),
            F.col("payment_group"),
            F.col("day_hour_bucket"),
            F.col("trip_distance_bin"),
        ),
    )
)

characterization_cols = ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]
expected_values = {
    "pu_macro_zone": {"airport", "manhattan", "outer_borough", "unknown"},
    "payment_group": {"flex", "credit", "cash", "other"},
    "day_hour_bucket": {"late_night", "weekend", "weekday_am", "weekday_pm_peak", "other"},
    "trip_distance_bin": {"short", "medium", "long"},
}

for col_name, expected in expected_values.items():
    observed = {row[col_name] for row in df_feat.select(col_name).distinct().collect()}
    assert observed == expected, f"{col_name}: valores observados {observed}, esperados {expected}"

print("Variables de caracterización construidas y validadas.")


Esta celda calcula las fracciones de muestreo por `stratum_id`, con objetivo de M cercano a 5 millones de viajes y protección para estratos raros.


In [ ]:
N_M_TARGET = 5_000_000
MIN_PER_STRATUM = 500
EXPECTED_M = 5_030_141

n_population = df_feat.count()

D_dist = (
    df_feat.groupBy("stratum_id")
    .count()
    .withColumnRenamed("count", "n_D")
    .withColumn("p_D", F.col("n_D") / F.lit(n_population))
)

n_strata_D = D_dist.count()
print(f"Estratos poblados en población analítica: {n_strata_D} de 240")
assert n_strata_D == 240, "Se esperaban 240 estratos poblados"

target = (
    D_dist
    .withColumn("target_n_raw", F.round(F.lit(N_M_TARGET) * F.col("p_D")).cast("long"))
    .withColumn(
        "target_n",
        F.least(
            F.col("n_D"),
            F.greatest(F.lit(MIN_PER_STRATUM).cast("long"), F.col("target_n_raw")),
        ),
    )
    .withColumn("fraction", F.col("target_n") / F.col("n_D"))
)

fractions_etapa2 = {
    row["stratum_id"]: float(row["fraction"])
    for row in target.select("stratum_id", "fraction").collect()
}

assert len(fractions_etapa2) == 240
assert all(0 < value <= 1.0 for value in fractions_etapa2.values())

expected_target_n = target.agg(F.sum("target_n").alias("n")).first()["n"]
print(f"Tamaño esperado de M por fracciones: {expected_target_n:,}")


Esta celda extrae M con `sampleBy`, materializa la muestra y valida que conserve los 240 estratos esperados.


In [ ]:
SEED_M = 42

M = df_feat.stat.sampleBy("stratum_id", fractions_etapa2, seed=SEED_M).cache()
n_M = M.count()

M_dist = (
    M.groupBy("stratum_id")
    .count()
    .withColumnRenamed("count", "n_M")
    .withColumn("p_M", F.col("n_M") / F.lit(n_M))
)

n_strata_M = M_dist.count()
print(f"Tamaño real de M: {n_M:,}")
print(f"Estratos poblados en M: {n_strata_M} de 240")

assert abs(n_M - EXPECTED_M) / EXPECTED_M < 0.02, "M difiere más de 2% del tamaño esperado"
assert n_strata_M == 240, "M debe conservar los 240 estratos"

repr_check = (
    D_dist.join(M_dist, "stratum_id", "inner")
    .withColumn("diff_pp", (F.col("p_M") - F.col("p_D")) * 100)
    .withColumn("abs_diff_pp", F.abs(F.col("diff_pp")))
)

repr_check.orderBy(F.desc("abs_diff_pp")).show(10, truncate=False)


**Lectura de M.** La muestra reconstruida contiene **5,029,725 filas** y conserva **240 de 240 estratos**. La mayor diferencia por estrato frente a la población analítica queda por debajo de 0.1 puntos porcentuales en la tabla mostrada, por lo que M conserva la estructura de la población después de la limpieza.


Esta celda compara las distribuciones marginales de las variables de caracterización entre la población analítica y M.


In [ ]:
def collect_distribution(df, column_name, total_rows):
    return {
        row[column_name]: row["count"] / total_rows
        for row in df.groupBy(column_name).count().collect()
    }

representativity_rows = []
for column_name in characterization_cols:
    p_population = collect_distribution(df_feat, column_name, n_population)
    p_sample = collect_distribution(M, column_name, n_M)
    for category in sorted(set(p_population) | set(p_sample)):
        p_d = p_population.get(category, 0.0)
        p_m = p_sample.get(category, 0.0)
        representativity_rows.append({
            "variable": column_name,
            "categoria": str(category),
            "pct_poblacion_analitica": round(p_d * 100, 4),
            "pct_M": round(p_m * 100, 4),
            "diff_pp_M_menos_P": round((p_m - p_d) * 100, 4),
        })

representativity_pdf = pd.DataFrame(representativity_rows)
display(representativity_pdf)

**Lectura de representatividad marginal.** Las distribuciones de macrozona, grupo de pago, bloque horario y rango de distancia son cercanas entre M y la población analítica. La mayor diferencia marginal visible está en `pu_macro_zone = manhattan`, con alrededor de **0.5330 puntos porcentuales**, lo cual es pequeño para una muestra de este tamaño.


Esta celda define la función auxiliar para calcular duración del viaje. Se usa después tanto en la regresión supervisada como en el clustering no supervisado.


In [ ]:
def add_trip_duration(df):
    return (
        df
        .withColumn(
            "trip_duration_min_raw",
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / F.lit(60.0),
        )
        .filter(F.col("trip_duration_min_raw").between(0.5, 720.0))
        .withColumn("trip_duration_min", F.col("trip_duration_min_raw").cast("float"))
        .drop("trip_duration_min_raw")
    )

# 1. Cálculo del valor K para validación cruzada

Esta sección responde primero si los folds tendrán tamaño suficiente para representar a M. La actividad usa dos preguntas heredadas del proyecto: una supervisada, donde Random Forest predice `fare_amount`, y una no supervisada, donde KMeans identifica arquetipos operativos de viaje. En ambos casos `K_FOLDS` controla los pliegues de validación cruzada; no debe confundirse con `KMEANS_CLUSTERS`, que es el número de grupos del modelo no supervisado seleccionado en Etapa 3.


## 1.1 Estabilidad empírica de M

Esta celda sigue el enfoque de los ejemplos del profesor: tomar una muestra diagnóstica, acumular observaciones por tamaño creciente, calcular métricas estadísticas y ubicar el punto donde los cambios dejan de ser relevantes. El punto de estabilización no se fija a ojo: se calcula por checkpoints y después se grafica como línea roja. La conversión a Pandas se limita a una muestra de 200,000 filas y tres columnas, y la propia celda reporta su tiempo y memoria para dejar explícito el costo.


Esta celda prepara la muestra diagnóstica acotada y reporta el costo real de convertirla a Pandas.

In [ ]:
CONVERGENCE_SAMPLE_N = 200_000

M_duration_for_conv = add_trip_duration(
    M.select("fare_amount", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime")
).cache()
n_M_duration_for_conv = M_duration_for_conv.count()

t0_to_pandas = time.time()
conv_pdf = (
    M_duration_for_conv
    .select("fare_amount", "trip_distance", "trip_duration_min")
    .sample(fraction=min(1.0, CONVERGENCE_SAMPLE_N / n_M_duration_for_conv), seed=42)
    .toPandas()
)
to_pandas_seconds = time.time() - t0_to_pandas
conv_pdf = conv_pdf.sample(frac=1.0, random_state=42).reset_index(drop=True)
conv_pdf_memory_mib = conv_pdf.memory_usage(deep=True).sum() / (1024 ** 2)

conv_read_summary_df = pd.DataFrame([{
    "filas_en_pandas": len(conv_pdf),
    "columnas_en_pandas": len(conv_pdf.columns),
    "memoria_mib_pandas": conv_pdf_memory_mib,
    "segundos_conversion_pandas": to_pandas_seconds,
}])
display(conv_read_summary_df.round(2))


Esta celda calcula medias y desviaciones estándar acumuladas, evalúa checkpoints y selecciona el primer punto que permanece estable hasta el final.

In [ ]:
STABILITY_CHECKPOINT_STEP = 5_000
MEAN_ABS_TOLERANCE = {
    "fare_amount": 0.10,
    "trip_distance": 0.05,
    "trip_duration_min": 0.10,
}
STD_REL_TOLERANCE_PCT = 2.50

n = np.arange(1, len(conv_pdf) + 1)
cols = ["fare_amount", "trip_distance", "trip_duration_min"]
running_stats = {}

for col in cols:
    x = conv_pdf[col].to_numpy()
    running_mean = np.cumsum(x) / n
    running_std = np.sqrt(np.maximum(np.cumsum(x ** 2) / n - running_mean ** 2, 0))
    running_stats[col] = {
        "mean": running_mean,
        "std": running_std,
        "mean_final": running_mean[-1],
        "std_final": running_std[-1],
    }

checkpoints = np.arange(STABILITY_CHECKPOINT_STEP, len(conv_pdf) + 1, STABILITY_CHECKPOINT_STEP)
if checkpoints[-1] != len(conv_pdf):
    checkpoints = np.append(checkpoints, len(conv_pdf))

stability_checkpoint_rows = []
for checkpoint in checkpoints:
    idx = int(checkpoint) - 1
    for col in cols:
        stats = running_stats[col]
        mean_abs_change = abs(stats["mean"][idx] - stats["mean_final"])
        std_rel_change_pct = abs(stats["std"][idx] - stats["std_final"]) / abs(stats["std_final"]) * 100
        cumple_media = mean_abs_change <= MEAN_ABS_TOLERANCE[col]
        cumple_desv_std = std_rel_change_pct <= STD_REL_TOLERANCE_PCT
        stability_checkpoint_rows.append({
            "n_referencia": int(checkpoint),
            "variable": col,
            "media_en_n": stats["mean"][idx],
            "media_final_200k": stats["mean_final"],
            "cambio_abs_media": mean_abs_change,
            "tolerancia_media_abs": MEAN_ABS_TOLERANCE[col],
            "cumple_media": cumple_media,
            "desv_std_en_n": stats["std"][idx],
            "desv_std_final_200k": stats["std_final"],
            "cambio_rel_desv_std_pct": std_rel_change_pct,
            "tolerancia_desv_std_rel_pct": STD_REL_TOLERANCE_PCT,
            "cumple_desv_std": cumple_desv_std,
            "cumple_variable": cumple_media and cumple_desv_std,
        })

stability_checkpoint_df = pd.DataFrame(stability_checkpoint_rows)
stability_checkpoint_summary_df = (
    stability_checkpoint_df
    .groupby("n_referencia", as_index=False)
    .agg(
        variables_estables=("cumple_variable", "sum"),
        max_cambio_abs_media=("cambio_abs_media", "max"),
        max_cambio_rel_desv_std_pct=("cambio_rel_desv_std_pct", "max"),
    )
)
stability_checkpoint_summary_df["todas_variables_estables"] = stability_checkpoint_summary_df["variables_estables"] == len(cols)

stable_flags = stability_checkpoint_summary_df["todas_variables_estables"].to_numpy()
stable_from_here = np.array([stable_flags[i:].all() for i in range(len(stable_flags))])
stability_checkpoint_summary_df["estable_desde_aqui"] = stable_from_here

POINT_ESTABILIZACION = int(
    stability_checkpoint_summary_df.loc[
        stability_checkpoint_summary_df["estable_desde_aqui"], "n_referencia"
    ].iloc[0]
)


Esta celda grafica la estabilidad acumulada y marca con línea roja el punto calculado por la regla anterior.

In [ ]:
MEAN_GRAPH_INDEX = 0
ST_DEV_GRAPH_INDEX = 1

x_ticks = np.arange(0, len(conv_pdf) + 1, 25_000)
stability_label = f"{POINT_ESTABILIZACION:,} instancias"

fig, ax = plt.subplots(len(cols), 2, figsize=(14, 12))
for j, col in enumerate(cols):
    running_mean = running_stats[col]["mean"]
    running_std = running_stats[col]["std"]

    ax[j, MEAN_GRAPH_INDEX].plot(n, running_mean, linewidth=1.3)
    ax[j, MEAN_GRAPH_INDEX].axvline(POINT_ESTABILIZACION, color="red", linestyle=":", linewidth=1.2, label=stability_label)
    ax[j, MEAN_GRAPH_INDEX].set_title(f"Media acumulada de {col}")
    ax[j, MEAN_GRAPH_INDEX].set_xlabel("Número de instancias")
    ax[j, MEAN_GRAPH_INDEX].set_ylabel("Media")
    ax[j, MEAN_GRAPH_INDEX].set_xticks(x_ticks)
    ax[j, MEAN_GRAPH_INDEX].ticklabel_format(style="plain", axis="x")
    ax[j, MEAN_GRAPH_INDEX].grid(True, which="major", linewidth=0.8, alpha=0.75)
    ax[j, MEAN_GRAPH_INDEX].minorticks_on()
    ax[j, MEAN_GRAPH_INDEX].grid(True, which="minor", linewidth=0.35, alpha=0.25)
    ax[j, MEAN_GRAPH_INDEX].legend(loc="best")

    ax[j, ST_DEV_GRAPH_INDEX].plot(n, running_std, color="tab:orange", linewidth=1.3)
    ax[j, ST_DEV_GRAPH_INDEX].axvline(POINT_ESTABILIZACION, color="red", linestyle=":", linewidth=1.2, label=stability_label)
    ax[j, ST_DEV_GRAPH_INDEX].set_title(f"Desviación estándar acumulada de {col}")
    ax[j, ST_DEV_GRAPH_INDEX].set_xlabel("Número de instancias")
    ax[j, ST_DEV_GRAPH_INDEX].set_ylabel("Desviación estándar")
    ax[j, ST_DEV_GRAPH_INDEX].set_xticks(x_ticks)
    ax[j, ST_DEV_GRAPH_INDEX].ticklabel_format(style="plain", axis="x")
    ax[j, ST_DEV_GRAPH_INDEX].grid(True, which="major", linewidth=0.8, alpha=0.75)
    ax[j, ST_DEV_GRAPH_INDEX].minorticks_on()
    ax[j, ST_DEV_GRAPH_INDEX].grid(True, which="minor", linewidth=0.35, alpha=0.25)
    ax[j, ST_DEV_GRAPH_INDEX].legend(loc="best")

plt.tight_layout()
plt.show()


Esta celda muestra la evidencia tabular del punto seleccionado: el checkpoint anterior, el elegido, el siguiente y el detalle por variable.

In [ ]:
stability_df = stability_checkpoint_df[
    stability_checkpoint_df["n_referencia"] == POINT_ESTABILIZACION
].reset_index(drop=True)

selected_pos = stability_checkpoint_summary_df.index[
    stability_checkpoint_summary_df["n_referencia"] == POINT_ESTABILIZACION
][0]
stability_threshold_evidence_df = stability_checkpoint_summary_df.iloc[
    max(0, selected_pos - 1): min(len(stability_checkpoint_summary_df), selected_pos + 2)
].copy()

display(stability_threshold_evidence_df.round(4))
display(stability_df.round(4))

M_duration_for_conv.unpersist()


**Lectura de estabilidad.** El método replica la lógica de los ejemplos de clase: se acumulan observaciones, se calculan métricas y se escoge el primer tamaño desde el cual agregar más datos cambia poco los patrones estadísticos. El valor seleccionado se obtiene en código como `POINT_ESTABILIZACION`, recorriendo checkpoints cada 5,000 observaciones sobre tres variables: `fare_amount`, `trip_distance` y `trip_duration_min`. En la corrida revisada, ese valor fue 50,000 observaciones; si se reejecuta el notebook, el valor válido es el que indiquen `stability_threshold_evidence_df` y la variable `POINT_ESTABILIZACION` en esa ejecución.

La tabla `conv_read_summary_df` confirma que Pandas se usa solo sobre una muestra diagnóstica pequeña, no sobre M completa. La tabla `stability_threshold_evidence_df` compara el checkpoint anterior, el elegido y el siguiente, para mostrar que el punto no fue elegido a ojo. La tabla `stability_df` audita la evidencia por variable usando nombres explícitos como `tolerancia_media_abs` y `tolerancia_desv_std_rel_pct`. Con ese umbral empírico, la decisión de k verifica que cada fold de prueba quede suficientemente por encima del tamaño mínimo estable antes de discutir costo computacional.


## 1.2 Selección de K_FOLDS

Esta celda compara candidatos razonables para validación cruzada. Se elige el menor k estándar que mantiene folds grandes, estratificación completa y costo manejable para ejecutar Random Forest y KMeans por fold.


In [ ]:
K_FOLDS = 5
K_CANDIDATES = [3, 4, 5, 6, 10]

k_candidate_df = pd.DataFrame([
    {
        "k_folds": k,
        "n_test_aprox": int(n_M // k),
        "multiplo_vs_estabilizacion": (n_M / k) / POINT_ESTABILIZACION,
    }
    for k in K_CANDIDATES
])

display(k_candidate_df.round(2))

assert K_FOLDS == 5
assert n_M / K_FOLDS > POINT_ESTABILIZACION


**Decisión de k.** Se usa `K_FOLDS = 5` porque es el punto donde se equilibran tamaño de prueba, medición de variabilidad y costo. Con `k=5`, cada fold tiene aproximadamente 1,005,945 filas, es decir, `20.12` veces el punto empírico de estabilización; por eso bajar a `k=3` o `k=4` solo agrandaría folds que ya son suficientemente estables, pero dejaría menos particiones para observar la variabilidad entre cortes. Subir a `k=6` o `k=10` sí daría más repeticiones de validación, pero no corrige un problema de tamaño del fold: incluso `k=10` queda `10.06` veces por encima del umbral de estabilización. La ganancia esperada sería marginal, mientras que el costo crece casi linealmente porque cada fold implica volver a entrenar RF y KMeans; `k=6` agrega 20% más corridas y `k=10` duplica el costo frente a `k=5`.


# 2. Construcción de los k-folds

Los folds se construyen sobre M con asignación aleatoria reproducible dentro de cada estrato. Así cada fold conserva la estructura de caracterización usada desde la Etapa 2.


## 2.1 Asignación estratificada de folds sobre M

Esta celda crea `fold_id` con orden aleatorio reproducible dentro de cada `stratum_id`. También crea un identificador técnico de fila solo para validar que la partición no duplique ni pierda registros.


In [ ]:
SEED_FOLDS = 123

w_fold = Window.partitionBy("stratum_id").orderBy(F.rand(seed=SEED_FOLDS))

M_folds = (
    M
    .withColumn("rn_fold", F.row_number().over(w_fold))
    .withColumn("fold_id", F.ntile(K_FOLDS).over(w_fold))
    .withColumn("fold_row_id", F.concat_ws("|", F.col("stratum_id"), F.col("rn_fold").cast("string")))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

n_M_folds = M_folds.count()
print(f"Filas en M_folds: {n_M_folds:,}")
assert n_M_folds == n_M, "M_folds debe conservar todas las filas de M"


## 2.2 Validación de los folds

Estas celdas verifican tamaño, ausencia de duplicados, cobertura de estratos y balance marginal de las variables de caracterización.


In [ ]:
fold_sizes = (
    M_folds.groupBy("fold_id")
    .count()
    .withColumnRenamed("count", "n_fold")
    .orderBy("fold_id")
    .withColumn("pct_M", F.round(F.col("n_fold") / F.lit(n_M) * 100, 4))
)
fold_sizes_df = fold_sizes.cache()
fold_sizes_pdf = fold_sizes_df.toPandas()
display(fold_sizes_pdf)

fold_partition_check = M_folds.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("fold_row_id").alias("n_distinct_fold_row_id"),
    F.countDistinct("fold_id").alias("n_folds"),
).toPandas()
display(fold_partition_check)

assert int(fold_partition_check.loc[0, "n_rows"]) == n_M
assert int(fold_partition_check.loc[0, "n_distinct_fold_row_id"]) == n_M
assert int(fold_partition_check.loc[0, "n_folds"]) == K_FOLDS


Esta celda valida que cada fold conserve la cobertura completa de estratos.


In [ ]:
strata_per_fold = (
    M_folds.groupBy("fold_id")
    .agg(F.countDistinct("stratum_id").alias("n_strata"))
    .orderBy("fold_id")
)
display(strata_per_fold.toPandas())

assert strata_per_fold.filter(F.col("n_strata") != n_strata_M).count() == 0, "Cada fold debe contener los 240 estratos"


Esta celda compara el balance marginal de las variables de caracterización entre M y cada fold.


In [ ]:
dist_balance_tables = []
for col_name in characterization_cols:
    base_dist = (
        M.groupBy(col_name)
        .count()
        .withColumnRenamed("count", "n_M_col")
        .withColumn("p_M_col", F.col("n_M_col") / F.lit(n_M))
    )
    fold_dist = (
        M_folds.groupBy("fold_id", col_name)
        .count()
        .join(fold_sizes_df, "fold_id")
        .withColumn("p_fold_col", F.col("count") / F.col("n_fold"))
    )
    diff = (
        fold_dist.join(base_dist, col_name, "left")
        .withColumn("abs_diff_pp", F.abs(F.col("p_fold_col") - F.col("p_M_col")) * 100)
    )
    dist_balance_tables.append(
        diff.groupBy("fold_id")
        .agg(F.round(F.max("abs_diff_pp"), 4).alias("max_abs_diff_pp"))
        .withColumn("variable", F.lit(col_name))
        .select("variable", "fold_id", "max_abs_diff_pp")
    )

dist_balance_df = reduce(lambda a, b: a.unionByName(b), dist_balance_tables)
display(dist_balance_df.orderBy("variable", "fold_id").toPandas())


**Lectura de folds.** Los cinco folds forman una partición completa de M: 5,029,725 filas, 5,029,725 identificadores técnicos únicos y cinco folds. Los tamaños quedaron prácticamente idénticos, entre 1,005,854 y 1,006,042 filas, con proporciones de 19.9982% a 20.0019%. Cada fold conserva los 240 estratos. El máximo desbalance marginal observado fue 0.0060 puntos porcentuales en `pu_macro_zone`, por lo que los folds son representativos de la población caracterizada y no simples cortes aleatorios sin control de estratos.


# 3. Experimentación con Validación Cruzada

Esta sección reentrena los modelos ganadores de Etapa 3 dentro de cada fold. La pregunta supervisada usa Random Forest para predecir tarifa; la pregunta no supervisada usa KMeans para validar la estabilidad de arquetipos operativos.


## 3.1 Base modelable común

Esta celda agrega duración del viaje, castea la variable binaria y libera la muestra cruda cuando ya existe la versión con folds lista para modelar.


In [ ]:
M_folds_ml = (
    add_trip_duration(M_folds)
    .withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
n_M_model = M_folds_ml.count()

print(f"Filas en M antes de filtro de duración: {n_M:,}")
print(f"Filas modelables tras filtro de duración: {n_M_model:,}")
print(f"Pérdida por filtro de duración: {(n_M - n_M_model) / n_M * 100:.3f}%")

M.unpersist()
M_folds.unpersist()


## 3.2 Pipeline supervisado: Random Forest

Esta celda solo prepara el modelo supervisado: define las variables predictoras, evita fuga de información por variables monetarias derivadas del objetivo y ensambla el pipeline de preprocesamiento más Random Forest. Aquí todavía no se pasan los folds al modelo; la separación de entrenamiento y prueba ocurre en la validación cruzada de la sección 3.4.


In [ ]:
TARGET = "fare_amount"
MODEL_SEED = 42
SUPERVISED_CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket", "cbd_period_flag"]
SUPERVISED_NUM_COLS = ["trip_distance", "trip_duration_min", "is_flex_fare", "passenger_count"]
SUPERVISED_FEATURE_COLS = SUPERVISED_NUM_COLS + SUPERVISED_CAT_COLS
LEAKAGE_COLS = {
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
}

assert TARGET not in SUPERVISED_FEATURE_COLS
assert not (set(SUPERVISED_FEATURE_COLS) & LEAKAGE_COLS), "Hay columnas con fuga en las features"
missing_supervised = sorted(set(SUPERVISED_FEATURE_COLS + [TARGET]) - set(M_folds_ml.columns))
assert not missing_supervised, f"Columnas faltantes para supervisado: {missing_supervised}"

supervised_indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    for col_name in SUPERVISED_CAT_COLS
]
supervised_encoder = OneHotEncoder(
    inputCols=[f"{col_name}_idx" for col_name in SUPERVISED_CAT_COLS],
    outputCols=[f"{col_name}_ohe" for col_name in SUPERVISED_CAT_COLS],
)
supervised_assembler = VectorAssembler(
    inputCols=SUPERVISED_NUM_COLS + [f"{col_name}_ohe" for col_name in SUPERVISED_CAT_COLS],
    outputCol="features",
    handleInvalid="keep",
)

supervised_preprocessing = supervised_indexers + [supervised_encoder, supervised_assembler]

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET,
    numTrees=50,
    maxDepth=10,
    subsamplingRate=1.0,
    featureSubsetStrategy="auto",
    seed=MODEL_SEED,
)

pipeline = Pipeline(stages=supervised_preprocessing + [rf])

ev_rmse = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="rmse")
ev_mae = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="mae")
ev_r2 = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="r2")

print("Features supervisadas:", SUPERVISED_FEATURE_COLS)
print("Pipeline supervisado ensamblado con RandomForestRegressor.")


## 3.3 Pipeline no supervisado: KMeans

Esta celda define las features operativas del clustering, excluye importes monetarios y ensambla el pipeline no supervisado con escalamiento de numéricas antes de KMeans.


In [ ]:
KMEANS_CLUSTERS = 5
KMEANS_MAX_ITER = 30
AVERAGE_SPEED_CAP_MPH = 40.0

CLUSTER_NUM_COLS = [
    "trip_distance",
    "trip_duration_min",
    "passenger_count",
    "average_speed_mph_capped",
]
CLUSTER_CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket"]
CLUSTER_BIN_COLS = ["is_flex_fare"]
CLUSTER_PROFILE_COLS = CLUSTER_NUM_COLS + CLUSTER_BIN_COLS + ["is_post_cbd_period", TARGET]


def add_cluster_features(df):
    return (
        df
        .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / F.lit(60.0)))
        .withColumn("average_speed_mph_capped", F.least(F.col("average_speed_mph"), F.lit(AVERAGE_SPEED_CAP_MPH)))
        .withColumn("is_post_cbd_period", F.when(F.col("cbd_period_flag") == "post_cbd", F.lit(1.0)).otherwise(F.lit(0.0)))
    )

missing_cluster_base = sorted(set(["trip_distance", "trip_duration_min", "passenger_count", "is_flex_fare"] + CLUSTER_CAT_COLS) - set(M_folds_ml.columns))
assert not missing_cluster_base, f"Columnas faltantes para clustering: {missing_cluster_base}"

cluster_indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    for col_name in CLUSTER_CAT_COLS
]
cluster_encoder = OneHotEncoder(
    inputCols=[f"{col_name}_idx" for col_name in CLUSTER_CAT_COLS],
    outputCols=[f"{col_name}_ohe" for col_name in CLUSTER_CAT_COLS],
)
cluster_num_assembler = VectorAssembler(
    inputCols=CLUSTER_NUM_COLS,
    outputCol="cluster_num_features",
    handleInvalid="keep",
)
cluster_scaler = StandardScaler(
    inputCol="cluster_num_features",
    outputCol="cluster_num_scaled",
    withMean=True,
    withStd=True,
)
cluster_assembler = VectorAssembler(
    inputCols=["cluster_num_scaled"] + [f"{col_name}_ohe" for col_name in CLUSTER_CAT_COLS] + CLUSTER_BIN_COLS,
    outputCol="features",
    handleInvalid="keep",
)
cluster_preprocessing = cluster_indexers + [cluster_encoder, cluster_num_assembler, cluster_scaler, cluster_assembler]

kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=KMEANS_CLUSTERS,
    seed=MODEL_SEED,
    maxIter=KMEANS_MAX_ITER,
)
cluster_pipeline = Pipeline(stages=cluster_preprocessing + [kmeans])
cluster_evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

print("Features no supervisadas numéricas:", CLUSTER_NUM_COLS)
print("Features no supervisadas categóricas:", CLUSTER_CAT_COLS)
print("Pipeline no supervisado ensamblado con KMeans.")


## 3.4 Validación cruzada supervisada

Esta celda es donde se pasa la data al modelo. En cada iteración, `fold_id == i` se usa como prueba y `fold_id != i` como entrenamiento. Después, `pipeline.fit(train_fold)` ajusta el pipeline con los cuatro folds de entrenamiento y `model_i.transform(test_fold)` evalúa el fold retenido.

PySpark sí incluye `CrossValidator` y podría usarse para la parte supervisada con una columna de folds definida por el usuario. Aquí se mantiene el loop explícito porque la actividad no solo requiere una métrica promedio: también se necesitan métricas individuales por fold, brecha train/test, errores por segmento y predicciones retenidas para las visualizaciones posteriores. El principio estadístico es el mismo que en `CrossValidator`; cambia el nivel de control y trazabilidad de la implementación.


In [ ]:
fold_sizes_model = {
    row["fold_id"]: row["count"]
    for row in M_folds_ml.groupBy("fold_id").count().collect()
}
n_M_model = sum(fold_sizes_model.values())

resultados = []
seg_rows = []
scatter_pdf = None

for i in range(1, K_FOLDS + 1):
    test_fold = M_folds_ml.filter(F.col("fold_id") == i)
    train_fold = M_folds_ml.filter(F.col("fold_id") != i)

    t0 = time.time()
    model_i = pipeline.fit(train_fold)
    fit_seconds = time.time() - t0

    pred_test = (
        model_i.transform(test_fold)
        .select(TARGET, "prediction", "is_flex_fare")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    pred_test.count()

    pred_train = (
        model_i.transform(train_fold.sample(fraction=0.25, seed=42 + i))
        .select(TARGET, "prediction")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    pred_train.count()

    resultados.append({
        "fold": i,
        "rmse_test": ev_rmse.evaluate(pred_test),
        "mae_test": ev_mae.evaluate(pred_test),
        "r2_test": ev_r2.evaluate(pred_test),
        "rmse_train": ev_rmse.evaluate(pred_train),
        "mae_train": ev_mae.evaluate(pred_train),
        "r2_train": ev_r2.evaluate(pred_train),
        "fit_seconds": fit_seconds,
        "n_train": n_M_model - fold_sizes_model[i],
        "n_test": fold_sizes_model[i],
    })

    for flex_val in (0, 1):
        seg = pred_test.filter(F.col("is_flex_fare") == flex_val)
        seg_n = seg.count()
        seg_rows.append({
            "fold": i,
            "segmento": "Flex Fare" if flex_val else "Metered",
            "n": seg_n,
            "rmse": ev_rmse.evaluate(seg) if seg_n else np.nan,
        })

    if i == 1:
        scatter_pdf = pred_test.sample(fraction=min(1.0, 5000 / fold_sizes_model[i]), seed=42).toPandas()

    pred_train.unpersist()
    pred_test.unpersist()

cv_df = pd.DataFrame(resultados)
seg_df = pd.DataFrame(seg_rows)

display(cv_df.round(4))
display(
    cv_df[["rmse_test", "mae_test", "r2_test", "rmse_train", "mae_train", "r2_train", "fit_seconds"]]
    .agg(["mean", "std"])
    .round(4)
)


**Lectura supervisada.** La validación cruzada ejecutada confirma estabilidad del Random Forest. El RMSE test promedio fue 5.1340 USD con desviación estándar 0.1466; el MAE test promedio fue 1.5747 USD con desviación estándar 0.0081; y el R2 test promedio fue 0.9173 con desviación estándar 0.0041. Por fold, el RMSE test quedó entre 4.9678 y 5.2913, MAE entre 1.5639 y 1.5831, y R2 entre 0.9130 y 0.9221. El tiempo de ajuste queda del orden de uno a dos minutos por fold según la carga local de Spark, bastante menor que el entrenamiento con búsqueda de hiperparámetros de Etapa 3 porque aquí se valida una configuración ya seleccionada.


## 3.5 Validación por folds no supervisada

Esta celda ajusta KMeans dentro de cada fold. La métrica principal es silhouette; además se mide balance de clusters y se conserva un perfil de arquetipos de un fold representativo.

Se muestran salidas compactas porque el objetivo de esta sección es validar el proceso, no leer manualmente todas las combinaciones fold-cluster. La tabla de métricas por fold queda completa porque resume la estabilidad del clustering. Para balance y perfiles se muestran solo los primeros registros como auditoría de forma y contenido; los DataFrames completos se conservan en memoria y se usan más abajo en las visualizaciones. En esta tarea, la comparación completa se entiende mejor gráficamente, porque el patrón de uniformidad o sensibilidad entre folds se percibe más claro por color, rango y agrupación visual que en una tabla larga de 25 filas.


In [ ]:
cluster_rows = []
cluster_balance_rows = []
cluster_profile_pdf = None
cluster_profile_rows = []
TRAIN_SILHOUETTE_SAMPLE_N = 500_000

for i in range(1, K_FOLDS + 1):
    test_fold = add_cluster_features(M_folds_ml.filter(F.col("fold_id") == i))
    train_fold = add_cluster_features(M_folds_ml.filter(F.col("fold_id") != i))

    t0 = time.time()
    cluster_model_i = cluster_pipeline.fit(train_fold)
    fit_seconds = time.time() - t0

    pred_train_cluster = (
        cluster_model_i.transform(train_fold)
        .select("features", "cluster")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    n_train_cluster = pred_train_cluster.count()
    pred_train_cluster_sample = (
        pred_train_cluster
        .sample(fraction=min(1.0, TRAIN_SILHOUETTE_SAMPLE_N / n_train_cluster), seed=240 + i)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    pred_train_cluster_sample.count()

    pred_test_cluster = (
        cluster_model_i.transform(test_fold)
        .select("features", "cluster", *CLUSTER_PROFILE_COLS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    n_test_cluster = pred_test_cluster.count()

    train_silhouette = cluster_evaluator.evaluate(pred_train_cluster_sample)
    test_silhouette = cluster_evaluator.evaluate(pred_test_cluster.select("features", "cluster"))

    test_counts_pdf = pred_test_cluster.groupBy("cluster").count().toPandas()
    test_counts_pdf["fold"] = i
    test_counts_pdf["fraction"] = test_counts_pdf["count"] / n_test_cluster
    test_counts_pdf = test_counts_pdf.rename(columns={"count": "n_test"})
    cluster_balance_rows.extend(test_counts_pdf.to_dict("records"))

    cluster_rows.append({
        "fold": i,
        "silhouette_train_sample": train_silhouette,
        "silhouette_test": test_silhouette,
        "fit_seconds": fit_seconds,
        "n_train_sample": pred_train_cluster_sample.count(),
        "n_test": n_test_cluster,
        "min_cluster_fraction_test": test_counts_pdf["fraction"].min(),
        "max_cluster_fraction_test": test_counts_pdf["fraction"].max(),
    })

    profile_exprs = [F.avg(F.col(col_name).cast("double")).alias(col_name) for col_name in CLUSTER_PROFILE_COLS]
    profile_fold_pdf = (
        pred_test_cluster
        .groupBy("cluster")
        .agg(F.count("*").alias("n"), *profile_exprs)
        .orderBy("cluster")
        .toPandas()
    )
    profile_fold_pdf.insert(0, "fold", i)
    cluster_profile_rows.extend(profile_fold_pdf.to_dict("records"))

    if i == 1:
        cluster_profile_pdf = profile_fold_pdf.drop(columns=["fold"]).copy()

    pred_train_cluster_sample.unpersist()
    pred_train_cluster.unpersist()
    pred_test_cluster.unpersist()

cluster_cv_df = pd.DataFrame(cluster_rows)
cluster_balance_df = pd.DataFrame(cluster_balance_rows)
cluster_profile_all_df = pd.DataFrame(cluster_profile_rows)

display(cluster_cv_df.round(4))

cluster_balance_preview_df = cluster_balance_df.sort_values(["fold", "cluster"]).head(10)
display(cluster_balance_preview_df.round(4))

display(cluster_profile_pdf.round(4))

cluster_profile_preview_df = cluster_profile_all_df.sort_values(["fold", "cluster"]).head(5)
display(cluster_profile_preview_df.round(4))

print("Las tablas completas alimentan las visualizaciones de las secciones 4.7 y 4.8.")


**Lectura no supervisada.** La validación por folds sugiere que KMeans recupera patrones operativos consistentes, pero con mayor sensibilidad que el modelo supervisado. La tabla de métricas muestra silhouettes test similares en los folds 1, 2, 3 y 5, entre 0.4257 y 0.4587, mientras que el fold 4 cae a 0.2777; esta diferencia debe corroborarse visualmente en la sección 4.6, donde la línea de silhouette evidencia el fold menos estable. El balance de clusters tampoco muestra colapso a un solo grupo, aunque sí hay un cluster dominante en varios folds; esto se revisa en la sección 4.7. Finalmente, los perfiles del fold 1 y el heatmap de todos los folds en la sección 4.8 permiten confirmar si los arquetipos interpretables se repiten por patrón operativo, no por el número local asignado al cluster.


## 3.6 Curva de complejidad supervisada

Esta celda entrena tres profundidades de Random Forest sobre una submuestra diagnóstica. Su objetivo no es volver a seleccionar modelo, sino visualizar sobreajuste en la sección 4.


In [ ]:
COMPLEXITY_FOLD = 1
COMPLEXITY_DEPTHS = [4, 8, 12]
train_complex_base = M_folds_ml.filter(F.col("fold_id") != COMPLEXITY_FOLD)
test_complex_base = M_folds_ml.filter(F.col("fold_id") == COMPLEXITY_FOLD)

train_complex_fraction = min(1.0, 250_000 / (n_M_model - fold_sizes_model[COMPLEXITY_FOLD]))
test_complex_fraction = min(1.0, 100_000 / fold_sizes_model[COMPLEXITY_FOLD])

train_complex = train_complex_base.sample(fraction=train_complex_fraction, seed=420).persist(StorageLevel.MEMORY_AND_DISK)
test_complex = test_complex_base.sample(fraction=test_complex_fraction, seed=421).persist(StorageLevel.MEMORY_AND_DISK)
train_complex.count()
test_complex.count()

complexity_rows = []
for depth in COMPLEXITY_DEPTHS:
    rf_depth = RandomForestRegressor(
        featuresCol="features",
        labelCol=TARGET,
        numTrees=50,
        maxDepth=depth,
        subsamplingRate=1.0,
        featureSubsetStrategy="auto",
        seed=MODEL_SEED,
    )
    pipeline_depth = Pipeline(stages=supervised_preprocessing + [rf_depth])
    t0 = time.time()
    model_depth = pipeline_depth.fit(train_complex)
    fit_seconds = time.time() - t0

    pred_train_depth = model_depth.transform(train_complex).select(TARGET, "prediction").persist(StorageLevel.MEMORY_AND_DISK)
    pred_test_depth = model_depth.transform(test_complex).select(TARGET, "prediction").persist(StorageLevel.MEMORY_AND_DISK)
    pred_train_depth.count()
    pred_test_depth.count()

    complexity_rows.append({
        "maxDepth": depth,
        "rmse_train": ev_rmse.evaluate(pred_train_depth),
        "rmse_test": ev_rmse.evaluate(pred_test_depth),
        "fit_seconds": fit_seconds,
    })

    pred_train_depth.unpersist()
    pred_test_depth.unpersist()

train_complex.unpersist()
test_complex.unpersist()

complexity_df = pd.DataFrame(complexity_rows)
display(complexity_df.round(4))


# 4. Resultados de la Validación Cruzada

Las gráficas muestran los resultados supervisados y no supervisados con foco en variabilidad, sobreajuste, segmentos y estabilidad de arquetipos.


## 4.1 Variabilidad supervisada por fold

Esta figura muestra RMSE, MAE y R2 en prueba por fold. Las líneas punteadas indican el promedio de cada métrica.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metric_specs = [
    ("rmse_test", "RMSE test", "Error en USD"),
    ("mae_test", "MAE test", "Error en USD"),
    ("r2_test", "R2 test", "R2"),
]
for ax, (metric, title, ylabel) in zip(axes, metric_specs):
    sns.lineplot(data=cv_df, x="fold", y=metric, marker="o", linewidth=2, ax=ax)
    mean_val = cv_df[metric].mean()
    ax.axhline(mean_val, color="red", linestyle=":", linewidth=1.5, label=f"Media {mean_val:.4f}")
    ax.set_title(title)
    ax.set_xlabel("Fold")
    ax.set_ylabel(ylabel)
    ax.set_xticks(sorted(cv_df["fold"].unique()))
    ax.legend(loc="best")
plt.tight_layout()
plt.show()


**Lectura.** El Random Forest muestra variación moderada entre folds, pero el desempeño se mantiene en una banda estrecha. El RMSE test promedio es 5.1340 USD, con desviación estándar de 0.1466; en términos relativos, esa desviación equivale a cerca de 2.9% del promedio. El RMSE por fold va de 4.9678 a 5.2913 USD y el R2 test se mantiene entre 0.9130 y 0.9221. En conjunto, la gráfica indica que el modelo conserva un desempeño estable al cambiar el fold de prueba.


## 4.2 Brecha train/test supervisada

Esta figura cuantifica la diferencia `RMSE_test - RMSE_train`. Una brecha positiva moderada es esperable porque el entrenamiento se mide en datos vistos por el modelo.


In [ ]:
gap_df = cv_df.assign(rmse_gap=cv_df["rmse_test"] - cv_df["rmse_train"])
plt.figure(figsize=(8, 4.5))
sns.barplot(data=gap_df, x="fold", y="rmse_gap", color="#4C78A8")
plt.axhline(gap_df["rmse_gap"].mean(), color="red", linestyle=":", linewidth=1.5, label=f"Media {gap_df['rmse_gap'].mean():.4f}")
plt.axhline(0, color="black", linewidth=1)
plt.title("Brecha de generalización por fold")
plt.xlabel("Fold")
plt.ylabel("RMSE test menos RMSE train")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

display(gap_df[["fold", "rmse_train", "rmse_test", "rmse_gap"]].round(4))


**Lectura.** La brecha `RMSE_test - RMSE_train` alterna signos entre folds y su promedio es -0.0188. Esto indica que no hay una señal sistemática de sobreajuste en la validación cruzada; las diferencias entre entrenamiento muestreado y prueba son pequeñas frente al RMSE promedio de 5.1340. El fold 1 y el fold 4 tienen brecha positiva, mientras que los folds 2, 3 y 5 tienen brecha negativa, compatible con variabilidad normal entre particiones.


## 4.3 Curva de complejidad

Esta figura compara profundidades del bosque en una submuestra diagnóstica. Sirve para observar cómo cambia el error de entrenamiento y prueba cuando aumenta la capacidad del modelo.


In [ ]:
complexity_long = complexity_df.melt(
    id_vars="maxDepth",
    value_vars=["rmse_train", "rmse_test"],
    var_name="conjunto",
    value_name="rmse",
)
plt.figure(figsize=(8, 5))
sns.lineplot(data=complexity_long, x="maxDepth", y="rmse", hue="conjunto", marker="o", linewidth=2)
plt.title("Curva de complejidad para detectar sobreajuste")
plt.xlabel("Profundidad máxima")
plt.ylabel("RMSE")
plt.tight_layout()
plt.show()

display(complexity_df.round(4))


**Lectura.** La curva diagnóstica mejora de `maxDepth=4` a `maxDepth=12`: el RMSE test baja de 6.7810 a 5.0929. Sin embargo, en profundidad 12 la brecha train/test se amplía, porque el RMSE train baja a 4.7113 mientras el test queda en 5.0929. Esto sugiere que la mayor complejidad todavía ayuda en esta submuestra, pero empieza a mostrar una separación que debe vigilarse.


## 4.4 Predicción, residuales y segmento de tarifa

Esta figura usa una muestra del fold 1 para revisar alineación entre tarifa real y predicha, y residuales coloreados por tipo de tarifa.


In [ ]:
scatter_plot_pdf = scatter_pdf.copy()
scatter_plot_pdf["segmento"] = np.where(scatter_plot_pdf["is_flex_fare"] == 1, "Flex Fare", "Metered")
scatter_plot_pdf["residual"] = scatter_plot_pdf["prediction"] - scatter_plot_pdf[TARGET]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(
    data=scatter_plot_pdf,
    x=TARGET,
    y="prediction",
    hue="segmento",
    alpha=0.35,
    s=18,
    ax=ax[0],
)
min_val = min(scatter_plot_pdf[TARGET].min(), scatter_plot_pdf["prediction"].min())
max_val = max(scatter_plot_pdf[TARGET].max(), scatter_plot_pdf["prediction"].max())
ax[0].plot([min_val, max_val], [min_val, max_val], color="black", linewidth=1)
ax[0].set_title("Tarifa real vs predicha")
ax[0].set_xlabel("fare_amount real")
ax[0].set_ylabel("Predicción")

sns.scatterplot(
    data=scatter_plot_pdf,
    x="prediction",
    y="residual",
    hue="segmento",
    alpha=0.35,
    s=18,
    ax=ax[1],
    legend=False,
)
ax[1].axhline(0, color="black", linewidth=1)
ax[1].set_title("Residuales vs predicción")
ax[1].set_xlabel("Predicción")
ax[1].set_ylabel("Predicción menos real")
plt.tight_layout()
plt.show()


**Lectura.** La nube principal se concentra cerca de la diagonal en tarifas bajas y medias, pero aparecen outliers con residuales grandes, especialmente en tarifas reales altas. Esto explica por qué RMSE es bastante mayor que MAE: el comportamiento típico es razonable, pero existe una cola de errores importantes. El segmento Flex Fare aparece más disperso, consistente con su mayor error por segmento.


## 4.5 RMSE por segmento y fold

Esta figura compara el error entre viajes con taxímetro y viajes Flex Fare para verificar si la calidad supervisada es homogénea.


In [ ]:
seg_pivot = seg_df.pivot(index="segmento", columns="fold", values="rmse")
plt.figure(figsize=(8, 4))
sns.heatmap(seg_pivot, annot=True, fmt=".2f", cmap="crest", square=True)
plt.title("RMSE por segmento y fold")
plt.xlabel("Fold")
plt.ylabel("Segmento")
plt.tight_layout()
plt.show()

display(seg_df.round(4))


**Lectura.** El heatmap muestra que Flex Fare concentra el error: su RMSE se mantiene entre 7.79 y 7.88 en los cinco folds, mientras Metered queda entre 4.25 y 4.70. Por eso la conclusión no debe limitarse al promedio global del modelo; el Random Forest generaliza bien, pero el régimen Flex sigue siendo el segmento más difícil.


## 4.6 Silhouette de KMeans por fold

Esta figura muestra la calidad no supervisada en entrenamiento muestreado y prueba. La cercanía entre ambas líneas indica estabilidad del agrupamiento.


In [ ]:
cluster_silhouette_long = cluster_cv_df.melt(
    id_vars="fold",
    value_vars=["silhouette_train_sample", "silhouette_test"],
    var_name="conjunto",
    value_name="silhouette",
)
plt.figure(figsize=(8, 4.5))
sns.lineplot(data=cluster_silhouette_long, x="fold", y="silhouette", hue="conjunto", marker="o", linewidth=2)
plt.axhline(cluster_cv_df["silhouette_test"].mean(), color="red", linestyle=":", linewidth=1.5, label="Media test")
plt.title("Silhouette de KMeans por fold")
plt.xlabel("Fold")
plt.ylabel("Silhouette")
plt.xticks(sorted(cluster_cv_df["fold"].unique()))
plt.tight_layout()
plt.show()

display(cluster_cv_df.round(4))


**Lectura.** Silhouette evalúa separación y cohesión de clusters. En esta ejecución, KMeans muestra estabilidad razonable en los folds 1, 2, 3 y 5, con silhouette test entre 0.4257 y 0.4587, pero el fold 4 cae a 0.2777. El promedio test es 0.4154 con desviación estándar 0.0782. Esto indica que los arquetipos son útiles, aunque menos estables que el desempeño supervisado del Random Forest.


## 4.7 Balance de clusters por fold

Esta figura muestra, para cada fold, la fracción del cluster más pequeño y del más grande en prueba. Ayuda a detectar segmentaciones degeneradas.


In [ ]:
cluster_balance_summary = (
    cluster_balance_df
    .groupby("fold")
    .agg(
        min_fraction=("fraction", "min"),
        max_fraction=("fraction", "max"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4.5))
for _, row in cluster_balance_summary.iterrows():
    ax.plot([row["min_fraction"], row["max_fraction"]], [row["fold"], row["fold"]], color="#9AA6B2", linewidth=2)
ax.scatter(cluster_balance_summary["min_fraction"], cluster_balance_summary["fold"], color="#4C78A8", label="Cluster menor")
ax.scatter(cluster_balance_summary["max_fraction"], cluster_balance_summary["fold"], color="#F58518", label="Cluster mayor")
ax.set_title("Rango de participación de clusters en prueba")
ax.set_xlabel("Fracción de filas del fold")
ax.set_ylabel("Fold")
ax.set_yticks(sorted(cluster_balance_summary["fold"].unique()))
ax.legend(loc="best")
plt.tight_layout()
plt.show()

display(cluster_balance_summary.round(4))


**Lectura.** El balance de clusters muestra un grupo dominante en casi todos los folds: la fracción máxima va de 0.4625 a 0.6570, y la fracción mínima promedio es 0.0454. No hay colapso total a un solo cluster, pero sí una estructura desbalanceada similar a la observada en Etapa 3, donde el arquetipo urbano regular concentraba una parte grande de los viajes.


## 4.8 Perfil de arquetipos no supervisados

Esta figura perfila los clusters del fold 1 como referencia interpretativa y luego ordena todos los perfiles por arquetipo inferido. El objetivo no es comparar `cluster 0` contra `cluster 0`, sino agrupar filas que comparten patrón operativo aunque KMeans les haya asignado IDs distintos. El color usa z-score por variable para facilitar comparación; las anotaciones del fold 1 muestran el valor promedio original. La variable `is_post_cbd_period` representa la fracción de viajes posteriores al inicio del cargo CBD.


In [ ]:
profile_cols = ["trip_distance", "trip_duration_min", "passenger_count", "average_speed_mph_capped", "is_flex_fare", "is_post_cbd_period", TARGET]
profile_for_heatmap = cluster_profile_pdf.set_index("cluster")[profile_cols].astype(float)
profile_std = profile_for_heatmap.std(axis=0).replace(0, np.nan)
profile_z = ((profile_for_heatmap - profile_for_heatmap.mean(axis=0)) / profile_std).fillna(0)

plt.figure(figsize=(11, 5))
sns.heatmap(profile_z, annot=profile_for_heatmap.round(2), fmt=".2f", cmap="vlag", center=0)
plt.title("Perfil de arquetipos KMeans en fold 1")
plt.xlabel("Variable")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

display(cluster_profile_pdf.round(4))


def infer_archetype(row):
    if row["is_flex_fare"] >= 0.80:
        return "Flex urbano"
    if row["trip_distance"] >= 12 and row[TARGET] >= 50:
        return "Largo/aeropuerto"
    if row["passenger_count"] >= 3:
        return "Pasajeros multiples"
    if row["trip_distance"] >= 7 and row["average_speed_mph_capped"] >= 18:
        return "Mediano rapido"
    return "Urbano regular corto"

archetype_order = {
    "Urbano regular corto": 0,
    "Flex urbano": 1,
    "Mediano rapido": 2,
    "Largo/aeropuerto": 3,
    "Pasajeros multiples": 4,
}

profile_all_plot = cluster_profile_all_df.copy()
profile_all_plot["arquetipo_inferido"] = profile_all_plot.apply(infer_archetype, axis=1)
profile_all_plot["orden_arquetipo"] = profile_all_plot["arquetipo_inferido"].map(archetype_order)
profile_all_plot["fold_cluster"] = profile_all_plot.apply(
    lambda row: f"{row['arquetipo_inferido']} | F{int(row['fold'])}-C{int(row['cluster'])}",
    axis=1,
)
profile_all_plot = profile_all_plot.sort_values(["orden_arquetipo", "fold", "cluster"]).reset_index(drop=True)

archetype_map_df = profile_all_plot[
    ["fold", "cluster", "arquetipo_inferido", "n", "trip_distance", "trip_duration_min", "passenger_count", "average_speed_mph_capped", "is_flex_fare", "fare_amount"]
].copy()

profile_all_values = profile_all_plot.set_index("fold_cluster")[profile_cols].astype(float)
profile_all_std = profile_all_values.std(axis=0).replace(0, np.nan)
profile_all_z = ((profile_all_values - profile_all_values.mean(axis=0)) / profile_all_std).fillna(0)

plt.figure(figsize=(12, 9))
sns.heatmap(profile_all_z, cmap="vlag", center=0, linewidths=0.25, linecolor="white")
plt.title("Perfiles por arquetipo inferido (IDs de cluster no alineados)")
plt.xlabel("Variable")
plt.ylabel("Arquetipo | fold-cluster")
plt.tight_layout()
plt.show()

display(archetype_map_df.round(4))


**Lectura.** Esta visualización permite pasar de números de cluster a patrones operativos interpretables. El primer heatmap muestra el perfil del fold 1: el grupo urbano regular corto concentra viajes de baja distancia, baja duración y tarifa baja; Flex urbano se distingue por una proporción muy alta de `is_flex_fare`; mediano rápido combina distancia intermedia con velocidad alta; largo/aeropuerto concentra las mayores distancias, duraciones y tarifas; y pasajeros múltiples se identifica por un `passenger_count` claramente superior al resto. La segunda vista confirma cuáles de esos patrones se repiten al cambiar el fold de entrenamiento. El eje Y debe leerse por bloques de `arquetipo_inferido`, no por número de cluster, porque `C0`, `C1`, `C2`, etc. son etiquetas locales de cada entrenamiento y pueden permutarse entre folds. Por eso `F5-C2` se compara con otros largo/aeropuerto como `F1-C3` y `F2-C1`, no con cualquier `C2`.

El hallazgo principal es que KMeans sí recupera estructura operacional recurrente: Flex urbano y pasajeros múltiples aparecen de forma limpia en los cinco folds, y urbano regular corto domina el volumen con perfiles muy parecidos entre particiones. El arquetipo largo/aeropuerto también aparece, pero se ve más fuerte en folds 1, 2 y 5, mientras en folds 3 y 4 queda más atenuado y se mezcla más con viajes medianos; esto coincide con la menor silhouette observada en esos folds. En conjunto, el heatmap no solo decora la sección: permite detectar estabilidad de arquetipos, permutación de etiquetas de cluster y sensibilidad del clustering en segmentos operativos específicos.


# 5. Discusión y conclusiones


**Resumen de resultados.** En el bloque supervisado, Random Forest obtuvo RMSE test promedio de 5.1340 USD, MAE test promedio de 1.5747 USD y R2 test promedio de 0.9173. La desviación estándar del RMSE test fue 0.1466 y la brecha promedio `RMSE_test - RMSE_train` fue -0.0188. Estos valores indican desempeño estable entre folds y no muestran una señal sistemática de sobreajuste.

En el bloque no supervisado, KMeans obtuvo silhouette test promedio de 0.4154 con desviación estándar de 0.0782. El cluster menor representa en promedio 4.54% del fold, lo que indica grupos desbalanceados pero no colapsados. La revisión de complejidad supervisada favoreció `maxDepth = 12` dentro de la submuestra diagnóstica evaluada.


## 5.1 Discusión supervisada

El Random Forest responde la pregunta de predicción de tarifa planteada desde Semana 5. La validación cruzada con `K_FOLDS = 5` muestra un desempeño alto y estable: en la ejecución actual se obtuvo RMSE test promedio de 5.1340 USD, MAE test promedio de 1.5747 USD y R2 test promedio de 0.9173. La desviación estándar del RMSE test fue 0.1466, equivalente a cerca de 2.9% del RMSE promedio; por eso la variabilidad entre folds es baja en relación con el nivel de error observado.

La brecha entre entrenamiento y prueba debe leerse junto con la curva de complejidad. En la submuestra diagnóstica, `maxDepth = 12` reduce el RMSE test frente a profundidades menores, pero también amplía la separación entre train y test. Por eso el modelo supervisado de Etapa 3 se valida aquí como una solución robusta para la pregunta de predicción, sin afirmar que sea completamente insensible a la partición.


## 5.2 Discusión no supervisada

KMeans responde una pregunta distinta: no predice tarifa, sino que identifica arquetipos operativos de viaje definidos por distancia, duración, velocidad, zona, tipo de tarifa y temporalidad. Por esa razón no se compara contra Random Forest. La validación por folds evalúa si los arquetipos de Etapa 3 se mantienen cuando el entrenamiento cambia, sin cambiar el modelo ni reabrir la selección de clusters.

La silhouette test resume la separación de los clusters en datos no vistos por cada ajuste. En esta ejecución, los folds 1, 2, 3 y 5 mantienen silhouettes test entre 0.4257 y 0.4587, mientras el fold 4 baja a 0.2777. El balance de clusters añade una condición práctica: los grupos deben ser suficientemente poblados para ser útiles como perfiles operativos. Finalmente, el heatmap de perfiles permite interpretar los clusters por sus características dominantes, evitando depender del número asignado al cluster, que puede cambiar entre folds. Por eso el resultado no se presenta como una estabilidad perfecta: KMeans recupera arquetipos recurrentes, pero con sensibilidad visible en una partición.


## 5.3 Conclusión general

La entrega valida las dos líneas del proyecto. Para la pregunta supervisada, el Random Forest mantiene errores bajos y variabilidad reducida entre folds, lo que respalda su uso para estimar `fare_amount` en la muestra M. Para la pregunta no supervisada, KMeans se evalúa con silhouette, balance y perfil de arquetipos, de modo que la conclusión no se limita a una métrica global sino que revisa estabilidad e interpretabilidad.

La elección de `K_FOLDS = 5` queda justificada por evidencia empírica de estabilización, tamaño de cada fold, cobertura de estratos y costo computacional. Con folds cercanos a un millón de filas, la validación es representativa para Big Data sin duplicar innecesariamente los entrenamientos que exigiría k=10.

Como contraste metodológico documentado aparte, se ejecutaron corridas exploratorias con más folds. Esas corridas no forman parte de la selección principal ni cambian los modelos de Etapa 3; solo verifican si las conclusiones cambian al aumentar el costo experimental. El resultado supervisado se mantiene prácticamente igual, mientras que KMeans muestra mayor sensibilidad a la partición. Esta evidencia refuerza que `K_FOLDS = 5` era suficiente para la entrega principal y que el costo adicional de más folds no modifica la conclusión central.


# Declaración de uso de inteligencia artificial

Google. (2026). *Gemini 3.5 Flash* [Modelo de lenguaje grande], utilizado para el proceso de aprendizaje del contenido de la semana y la validación de errores conceptuales y de código. https://deepmind.google/models/gemini/flash/
